# **Part 0 : Deliverable Guidelines**

The deliverable must consist of:
- One single `.ipynb` notebook (Python notebook)
- A PDF version exported from the notebook
- If possible, the dataset used should also be attached, or the data source should be clearly referenced in the notebook

https://huggingface.co/datasets/FinanceInc/auditor_sentiment

## **Timeline**

The final deliverable is expected **??**.



**Data Requirements**
The data must be taken from **Hugging Face**, and the topic must be related to:
- Finance
- Insurance
- Risk

## **Expectations**

You are expected to clearly explain the context and justify **why this use case is relevant**.

## **Why this problem?**


Explain:
- What the problem is
- Why it matters
- Why it is worth studying

## **What is the data?**

Explain:
- What the dataset is
- Why this dataset was chosen
- How the data is prepared
- Why the data is suitable for the problem

## **Supervised Model**


You must build a **supervised learning model** using the chosen data type, such as:
- Image
- Audio
- Text
- etc.

Minimum expected:
- Decision Tree  
**or**
- Random Forest

## **Study and Interpretation of Results ⭐ (Most Important)**


You are expected to analyze and interpret the model results:

- Is the model performance satisfactory?
- How can the results be interpreted?
- Why are the results good or bad?
- What could have been improved?
- What are the limitations of the results?
- Are there any constraints such as:
  - High model cost
  - Ethical risks
  - Bias
  - Practical limitations

## **Evaluation Criteria**

- **6 to 8 points** → Quality of explanations  
- **2 points** → Quality of code  
- **4 to 6 points** → Depth of the work  
- **6 to 8 points** → Respect of instructions and requirements  

# **Part 1 : Data Preperation**

## **1.1 - Import Data Set**

We first found and imported our auditor_sentiment data set from hungging face by connecting to our account with a token.

In [22]:
from datasets import load_dataset
ds = load_dataset("Tydyannes69/auditor_sentiment")

Then we wanted to have a look at the data set to see if the importation functioned. So we decided to create a CSV file to be able to visulize it. The data base was already separated in two different data set. A "Train" and "Test" data set. We decided to keep it this way, so we could train our model on the "Train" data set and then measure the quality of our trained model on the "Test" data set.

In [23]:
import pandas as pd

df_train = ds["train"].to_pandas()
df_test = ds["test"].to_pandas()

print(df_train.head())
print(df_test.columns)

                                            sentence  label
0  Altia 's operating profit jumped to EUR 47 mil...      2
1  The agreement was signed with Biohit Healthcar...      2
2  Kesko pursues a strategy of healthy , focused ...      2
3  Vaisala , headquartered in Helsinki in Finland...      1
4  Also , a six-year historic analysis is provide...      1
Index(['sentence', 'label'], dtype='object')


Before doing any descriptive statistics we wanted to combine the "train" and "test" dataset to be able to study our entire dataset

In [24]:
df_combined = pd.concat([df_train, df_test], ignore_index=True)

ydata-cleaning

In [25]:
#pip install ydata-profiling
from ydata_profiling import ProfileReport
profile = ProfileReport(df_combined, title="Profiling Dataset", explorative=True)
profile.to_file("report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 627.70it/s]


Doublons

In [26]:
df_combined = df_combined.drop_duplicates()

Sepération

In [27]:
from sklearn.model_selection import train_test_split

df_train_original, df_test_original = train_test_split(df_combined, test_size=0.2,stratify=df_combined["label"],random_state=42)
df_train_original.to_csv("train.csv", index=False)
df_test_original.to_csv("test.csv", index=False)

## **1.2 - Descriptive Statistics**

In [28]:
# Nombre de lignes et colonnes
num_rows = df_combined.shape[0]
num_cols = df_combined.shape[1]

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")
print(f"Names of the Columns: {ds['train'].column_names}")

Number of rows: 4840
Number of columns: 2
Names of the Columns: ['sentence', 'label']


### Transformation of the variable "label"

In [29]:
df_combined["label_num"] = df_combined["label"].map({
    0: "negative",
    1: "neutral",
    2: "positive"
})

C:\Users\speed\AppData\Local\Temp\ipykernel_11196\1716255520.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_combined["label_num"] = df_combined["label"].map({


### Description of the variable "label"

In [30]:
print("Distribution des classes :")
print(df_combined["label_num"].value_counts())

print("\nProportions (%) :")
print((df_combined["label_num"].value_counts(normalize=True) * 100).round(2))

Distribution des classes :
label_num
neutral     2873
positive    1363
negative     604
Name: count, dtype: int64

Proportions (%) :
label_num
neutral     59.36
positive    28.16
negative    12.48
Name: proportion, dtype: float64


In [31]:
import matplotlib.pyplot as plt

df_combined["label_num"].value_counts().plot(kind="bar")
plt.title("Distribution des sentiments")
plt.xlabel("Sentiment")
plt.ylabel("Nombre")
plt.xticks(rotation=0)
plt.show()

C:\Users\speed\AppData\Local\Temp\ipykernel_11196\3857146993.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## **1.3 - Data Cleaning**

### Text Cleaning

We used the libraries NLTK and SpaCy for text processing and analysis. SpaCy (en_core_web_sm) facilitates advanced tasks such as lemmatization and syntactic analysis. Common words (stopwords) are removed in order to focus on and analyze the most meaningful terms.

Before converting raw text into usable data with CountVectorizer and TfidfVectorizer, it is necessary to clean the data. We therefore decided to standardize the text by converting it to lowercase, then removing special characters, numbers, and extra spaces.

In [32]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

# Stopwords anglais
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # suppression des stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\speed\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Lemmatize Text

Lemmatization reduces each word to its base form (e.g., “studying” becomes “study”). This reduces the complexity of the vocabulary while preserving the meaning of sentences.

We chose to lemmatize the textual variable "sentence", as it contains longer and meaningful sentences that are important for predicting our target variable.

We chose lemmatization rather than stemming because lemmatization preserves the meaning of words.

Finally, we applied lemmatization before vectorization to prevent the model from considering words like “study”, “studies”, and “studying” as three different terms. By doing so, we reduce computation time and improve the relevance of the results.

In [33]:
import spacy

# Charger modèle anglais
nlp = spacy.load("en_core_web_sm")

def lemmatize_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_punct]
    return " ".join(tokens)

Data cleaning and lemmatizing text on train et test dataset

In [52]:
df_train = df_train_original.copy()
df_test = df_test_original.copy()

df_train["sentence_clean"] = df_train_original["sentence"].apply(clean_text).apply(lemmatize_text)
df_test["sentence_clean"] = df_test_original["sentence"].apply(clean_text).apply(lemmatize_text)

In [53]:
df_train.head()

,sentence,label,sentence_clean
883,Currently Alfred A Palmberg is putting the fin...,1,currently alfre palmberg put finish touch one ...
3204,Finnish Rautaruukki has been awarded a contrac...,2,finnish rautaruukki award contract supply inst...
2422,Finnish Bore that is owned by the Rettig famil...,2,finnish bore own rettig family grow recently a...
4831,YIT 's Baltic sales in the first three quarter...,0,yit baltic sale first three quarter total mill...
3371,HKScan is one of the leading food companies in...,1,hkscan one lead food company northern europe h...


### Vectorizing Text

In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(df_train["sentence_clean"])
X_test_tfidf = tfidf_vectorizer.transform(df_test["sentence_clean"])

print(tfidf_vectorizer.get_feature_names_out()[:20])
print(X_train_tfidf)
print(X_test_tfidf)

['aaland' 'aalborg' 'aalto' 'aaron' 'aava' 'aazhang' 'ab' 'abb' 'abbott'
 'abc' 'aberration' 'abidjan' 'ability' 'able' 'abloy' 'abn'
 'abovementione' 'abp' 'abroad' 'absentee']
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 43281 stored elements and shape (3872, 6975)>
  Coords	Values
  (0, 1339)	0.20357660840403582
  (0, 178)	0.2993196260087947
  (0, 4351)	0.28515591975638815
  (0, 4858)	0.2339338365386516
  (0, 2100)	0.2555581127864836
  (0, 6324)	0.26094290399051395
  (0, 4200)	0.17271415661892042
  (0, 2103)	0.12982851564767553
  (0, 605)	0.21888572867418662
  (0, 1924)	0.2751066102429205
  (0, 5100)	0.28515591975638815
  (0, 528)	0.28515591975638815
  (0, 5021)	0.28515591975638815
  (0, 4788)	0.16673560611508106
  (0, 203)	0.14193875506909667
  (0, 1574)	0.2367298882246398
  (0, 6717)	0.28515591975638815
  (1, 2109)	0.14719003677292591
  (1, 4941)	0.2776199561310213
  (1, 459)	0.2626546711569255
  (1, 1212)	0.19577225581627095
  (1, 5992)	0.2397901304689967
  (1, 2

Top 5 words that come up the most

In [55]:
import numpy as np

# moyenne des scores TF-IDF pour chaque mot (colonne)
mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).ravel()

# récupérer les mots
feature_names = tfidf_vectorizer.get_feature_names_out()

# trier du plus important au moins important
top_indices = mean_tfidf.argsort()[::-1]

# afficher les 5 mots les plus importants
for i in top_indices[:5]:
    print(feature_names[i], mean_tfidf[i])

eur 0.04710254057734286
mn 0.030411858374001226
company 0.02602476554162505
sale 0.019654471195201063
profit 0.018840225484002684


### Dimension Reduction

In [56]:
from sklearn.decomposition import TruncatedSVD

# Réduction de dimension
svd = TruncatedSVD(n_components=170, random_state=42)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

print(X_train_svd.shape)
print(X_test_svd.shape)

(3872, 170)
(968, 170)


# **Part 2 : Classifier**

## **2.1 - Logistic Regression**

Training Model

In [57]:
y_train = df_train["label"]
y_test = df_test["label"]

from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_svd, y_train)

y_pred = clf.predict(X_test_svd)

Testing Model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6962809917355371
              precision    recall  f1-score   support

           0       0.76      0.28      0.41       121
           1       0.70      0.94      0.80       575
           2       0.66      0.36      0.47       272

    accuracy                           0.70       968
   macro avg       0.71      0.53      0.56       968
weighted avg       0.70      0.70      0.66       968



## **2.2 - Gaussian Classifier**

In [59]:
# Modèle Gaussian Naive Bayes
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()

# Entraînement
gnb.fit(X_train_svd, y_train)

# Prédiction
y_pred_gnb = gnb.predict(X_test_svd)

# Évaluation
print("Accuracy :", accuracy_score(y_test, y_pred_gnb))
print("\nClassification report :")
print(classification_report(y_test, y_pred_gnb))
print("\nConfusion matrix :")
print(confusion_matrix(y_test, y_pred_gnb))

Accuracy : 0.6229338842975206

Classification report :
              precision    recall  f1-score   support

           0       0.38      0.41      0.40       121
           1       0.68      0.83      0.75       575
           2       0.57      0.27      0.36       272

    accuracy                           0.62       968
   macro avg       0.54      0.51      0.50       968
weighted avg       0.61      0.62      0.60       968


Confusion matrix :
[[ 50  61  10]
 [ 50 480  45]
 [ 31 168  73]]


## **2.3 - Bernoulli Classifier**

In [60]:
from sklearn.naive_bayes import BernoulliNB

# Transformer TF-IDF en binaire
X_train_bin = (X_train_tfidf > 0).astype(int)
X_test_bin = (X_test_tfidf > 0).astype(int)

# Modèle Bernoulli
bnb = BernoulliNB()

# Entraînement
bnb.fit(X_train_bin, y_train)

# Prédiction
y_pred_bnb = bnb.predict(X_test_bin)

# Évaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy :", accuracy_score(y_test, y_pred_bnb))
print("\nClassification report :")
print(classification_report(y_test, y_pred_bnb))
print("\nConfusion matrix :")
print(confusion_matrix(y_test, y_pred_bnb))

Accuracy : 0.6921487603305785

Classification report :
              precision    recall  f1-score   support

           0       0.92      0.09      0.17       121
           1       0.71      0.94      0.81       575
           2       0.60      0.43      0.50       272

    accuracy                           0.69       968
   macro avg       0.74      0.49      0.49       968
weighted avg       0.71      0.69      0.64       968


Confusion matrix :
[[ 11  64  46]
 [  0 542  33]
 [  1 154 117]]


## **2.4 - Decision Tree**

In [64]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

best_score = 0
best_params = None

for max_depth in [5, 10, 15, 20, None]:
    for min_samples_split in [2, 5, 10, 20]:
        for min_samples_leaf in [1, 2, 4, 8]:
            for criterion in ["gini", "entropy"]:

                dt = DecisionTreeClassifier(
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    min_samples_leaf=min_samples_leaf,
                    criterion=criterion,
                    random_state=42
                )

                dt.fit(X_train_svd, y_train)
                y_pred = dt.predict(X_test_svd)
                acc = accuracy_score(y_test, y_pred)

                if acc > best_score:
                    best_score = acc
                    best_params = {
                        "max_depth": max_depth,
                        "min_samples_split": min_samples_split,
                        "min_samples_leaf": min_samples_leaf,
                        "criterion": criterion
                    }
                    best_model = dt

In [68]:
# Prédictions finales avec le meilleur modèle
y_pred_best = best_model.predict(X_test_svd)

print(best_params)
print("Accuracy:", accuracy_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))

{'max_depth': 10, 'min_samples_split': 20, 'min_samples_leaf': 1, 'criterion': 'gini'}
Accuracy: 0.6301652892561983
              precision    recall  f1-score   support

           0       0.42      0.29      0.34       121
           1       0.69      0.85      0.76       575
           2       0.50      0.31      0.38       272

    accuracy                           0.63       968
   macro avg       0.53      0.48      0.50       968
weighted avg       0.60      0.63      0.60       968



## **2.5 - Random Forest**

In [69]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

best_score = 0
best_params = None
best_model = None

for n_estimators in [50, 100, 200]:
    for max_depth in [5, 10, 20, None]:
        for max_features in ["sqrt", "log2"]:
            for min_samples_split in [2, 5, 10]:
                for min_samples_leaf in [1, 2, 4]:

                    rf = RandomForestClassifier(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        max_features=max_features,
                        min_samples_split=min_samples_split,
                        min_samples_leaf=min_samples_leaf,
                        random_state=42,
                        n_jobs=-1
                    )

                    rf.fit(X_train_svd, y_train)
                    y_pred = rf.predict(X_test_svd)
                    acc = accuracy_score(y_test, y_pred)

                    if acc > best_score:
                        best_score = acc
                        best_params = {
                            "n_estimators": n_estimators,
                            "max_depth": max_depth,
                            "max_features": max_features,
                            "min_samples_split": min_samples_split,
                            "min_samples_leaf": min_samples_leaf
                        }
                        best_model = rf

In [70]:
y_pred_best = best_model.predict(X_test_svd)

print(best_params)
print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred_best))

{'n_estimators': 100, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 5, 'min_samples_leaf': 1}
Accuracy: 0.6962809917355371

Classification report:

              precision    recall  f1-score   support

           0       0.78      0.26      0.39       121
           1       0.69      0.97      0.80       575
           2       0.73      0.32      0.44       272

    accuracy                           0.70       968
   macro avg       0.73      0.51      0.54       968
weighted avg       0.71      0.70      0.65       968



# **Part 3 : Classifier 2.0**

## **3.1 - Import Data Set**

Utilisation de df_train_original et df_test_original

## **3.2 - Data Cleaning**

### Text Cleaning

In [71]:
nltk.download('stopwords')

# Stopwords anglais
stop_words = set(stopwords.words('english'))

# 🔥 Liste des négations à garder
negations = {"not", "no", "never"}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = text.split()
    
    # garder les négations même si elles sont dans stopwords
    tokens = [word for word in tokens if (word not in stop_words or word in negations)]
    
    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\speed\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Lemmatize Text

In [72]:
import spacy

# Charger modèle anglais
nlp = spacy.load("en_core_web_sm")

def lemmatize_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_punct]
    return " ".join(tokens)

In [73]:
df_train["sentence_clean"] = df_train_original["sentence"].apply(clean_text).apply(lemmatize_text)
df_test["sentence_clean"] = df_test_original["sentence"].apply(clean_text).apply(lemmatize_text)

In [74]:
negations = {"not", "no", "never"}

# Compter combien de fois ils apparaissent
count = 0

for text in df_train["sentence_clean"]:
    words = text.split()
    count += sum(1 for w in words if w in negations)

print("Nombre de négations :", count)

Nombre de négations : 133


### Vectorizing Text

In [75]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(df_train["sentence_clean"])
X_test_tfidf = tfidf_vectorizer.transform(df_test["sentence_clean"])

print(tfidf_vectorizer.get_feature_names_out()[:20])
print(X_train_tfidf)
print(X_test_tfidf)

['aaland' 'aalborg' 'aalto' 'aaron' 'aava' 'aazhang' 'ab' 'abb' 'abbott'
 'abc' 'aberration' 'abidjan' 'ability' 'able' 'abloy' 'abn'
 'abovementione' 'abp' 'abroad' 'absentee']
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 43407 stored elements and shape (3872, 6977)>
  Coords	Values
  (0, 1339)	0.20357660840403582
  (0, 178)	0.2993196260087947
  (0, 4353)	0.28515591975638815
  (0, 4860)	0.2339338365386516
  (0, 2100)	0.2555581127864836
  (0, 6326)	0.26094290399051395
  (0, 4202)	0.17271415661892042
  (0, 2103)	0.12982851564767553
  (0, 605)	0.21888572867418662
  (0, 1924)	0.2751066102429205
  (0, 5102)	0.28515591975638815
  (0, 528)	0.28515591975638815
  (0, 5023)	0.28515591975638815
  (0, 4790)	0.16673560611508106
  (0, 203)	0.14193875506909667
  (0, 1574)	0.2367298882246398
  (0, 6719)	0.28515591975638815
  (1, 2109)	0.14719003677292591
  (1, 4943)	0.2776199561310213
  (1, 459)	0.2626546711569255
  (1, 1212)	0.19577225581627095
  (1, 5994)	0.2397901304689967
  (1, 2

Top 5 most used words

In [76]:
import numpy as np

# moyenne des scores TF-IDF pour chaque mot (colonne)
mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).ravel()

# récupérer les mots
feature_names = tfidf_vectorizer.get_feature_names_out()

# trier du plus important au moins important
top_indices = mean_tfidf.argsort()[::-1]

# afficher les 5 mots les plus importants
for i in top_indices[:5]:
    print(feature_names[i], mean_tfidf[i])

eur 0.04710185951243433
mn 0.03041197961312809
company 0.025960572940767385
sale 0.0196156801181617
profit 0.018832371625958844


### Dimension Reduction

In [77]:
from sklearn.decomposition import TruncatedSVD

# Réduction de dimension
svd = TruncatedSVD(n_components=170, random_state=42)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

print(X_train_svd.shape)
print(X_test_svd.shape)

(3872, 170)
(968, 170)


## **3.3 - Logistic Regression 2.0**

Training

In [78]:
y_train = df_train["label"]
y_test = df_test["label"]

from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_svd, y_train)

y_pred = clf.predict(X_test_svd)

Testing

In [81]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6973140495867769
              precision    recall  f1-score   support

           0       0.72      0.28      0.40       121
           1       0.70      0.94      0.80       575
           2       0.67      0.37      0.48       272

    accuracy                           0.70       968
   macro avg       0.70      0.53      0.56       968
weighted avg       0.70      0.70      0.66       968



## **3.4 - Gaussian Classifier 2.0**

In [82]:
# Modèle Gaussian Naive Bayes
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()

# Entraînement
gnb.fit(X_train_svd, y_train)

# Prédiction
y_pred_gnb = gnb.predict(X_test_svd)

# Évaluation
print("Accuracy :", accuracy_score(y_test, y_pred_gnb))
print("\nClassification report :")
print(classification_report(y_test, y_pred_gnb))
print("\nConfusion matrix :")

Accuracy : 0.609504132231405

Classification report :
              precision    recall  f1-score   support

           0       0.34      0.36      0.35       121
           1       0.67      0.83      0.74       575
           2       0.54      0.25      0.34       272

    accuracy                           0.61       968
   macro avg       0.52      0.48      0.48       968
weighted avg       0.59      0.61      0.58       968


Confusion matrix :


## **3.5 - Bernoulli Classifier 2.0**

In [83]:
from sklearn.naive_bayes import BernoulliNB

# Transformer TF-IDF en binaire
X_train_bin = (X_train_tfidf > 0).astype(int)
X_test_bin = (X_test_tfidf > 0).astype(int)

# Modèle Bernoulli
bnb = BernoulliNB()

# Entraînement
bnb.fit(X_train_bin, y_train)

# Prédiction
y_pred_bnb = bnb.predict(X_test_bin)

# Évaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy :", accuracy_score(y_test, y_pred_bnb))
print("\nClassification report :")
print(classification_report(y_test, y_pred_bnb))

Accuracy : 0.6921487603305785

Classification report :
              precision    recall  f1-score   support

           0       0.92      0.09      0.17       121
           1       0.71      0.94      0.81       575
           2       0.59      0.43      0.50       272

    accuracy                           0.69       968
   macro avg       0.74      0.49      0.49       968
weighted avg       0.71      0.69      0.64       968



## **3.6 - Decision Tree 2.0**

In [84]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

best_score = 0
best_params = None

for max_depth in [5, 10, 15, 20, None]:
    for min_samples_split in [2, 5, 10, 20]:
        for min_samples_leaf in [1, 2, 4, 8]:
            for criterion in ["gini", "entropy"]:

                dt = DecisionTreeClassifier(
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    min_samples_leaf=min_samples_leaf,
                    criterion=criterion,
                    random_state=42
                )

                dt.fit(X_train_svd, y_train)
                y_pred = dt.predict(X_test_svd)
                acc = accuracy_score(y_test, y_pred)

                if acc > best_score:
                    best_score = acc
                    best_params = {
                        "max_depth": max_depth,
                        "min_samples_split": min_samples_split,
                        "min_samples_leaf": min_samples_leaf,
                        "criterion": criterion
                    }
                    best_model = dt

In [ ]:
# Prédictions finales avec le meilleur modèle

y_pred_best = best_model.predict(X_test_svd)

print(best_params)
print("Accuracy:", accuracy_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))

{'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1, 'criterion': 'entropy'}
Accuracy: 0.628099173553719
              precision    recall  f1-score   support

           0       0.41      0.09      0.15       121
           1       0.66      0.94      0.77       575
           2       0.47      0.20      0.28       272

    accuracy                           0.63       968
   macro avg       0.51      0.41      0.40       968
weighted avg       0.57      0.63      0.56       968



## **3.7 - Random Forest 2.0**

In [87]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

best_score = 0
best_params = None
best_model = None

for n_estimators in [50, 100, 200]:
    for max_depth in [5, 10, 20, None]:
        for max_features in ["sqrt", "log2"]:
            for min_samples_split in [2, 5, 10]:
                for min_samples_leaf in [1, 2, 4]:

                    rf = RandomForestClassifier(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        max_features=max_features,
                        min_samples_split=min_samples_split,
                        min_samples_leaf=min_samples_leaf,
                        random_state=42,
                        n_jobs=-1
                    )

                    rf.fit(X_train_svd, y_train)
                    y_pred = rf.predict(X_test_svd)
                    acc = accuracy_score(y_test, y_pred)

                    if acc > best_score:
                        best_score = acc
                        best_params = {
                            "n_estimators": n_estimators,
                            "max_depth": max_depth,
                            "max_features": max_features,
                            "min_samples_split": min_samples_split,
                            "min_samples_leaf": min_samples_leaf
                        }
                        best_model = rf

In [88]:
y_pred_best = best_model.predict(X_test_svd)

print(best_params)
print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred_best))

{'n_estimators': 100, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'min_samples_leaf': 2}
Accuracy: 0.7014462809917356

Classification report:

              precision    recall  f1-score   support

           0       0.74      0.26      0.39       121
           1       0.69      0.98      0.81       575
           2       0.79      0.32      0.45       272

    accuracy                           0.70       968
   macro avg       0.74      0.52      0.55       968
weighted avg       0.72      0.70      0.65       968



# **Part 4 : Classifier 3.0**

## **4.1 - Import Data Set**

Utilisation de df_train_original et df_test_original

## **4.2 - Data Cleaning**

### Text Cleaning

In [89]:
def clean_text_bert(text):
    text = str(text).lower().strip()
    return text

In [90]:
df_train["sentence_clean"] = df_train_original["sentence"].apply(clean_text_bert)
df_test["sentence_clean"] = df_test_original["sentence"].apply(clean_text_bert)

## **4.3 - Sentence BERT Classifier**

In [ ]:
# pip install sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1) modèle d'embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# 2) transformer les phrases en vecteurs denses
X_train_emb = embedder.encode(df_train["sentence"].tolist(), show_progress_bar=True)
X_test_emb = embedder.encode(df_test["sentence"].tolist(), show_progress_bar=True)

# 3) labels
y_train = df_train["label"]
y_test = df_test["label"]

# 4) classifieur
clf = LogisticRegression(max_iter=2000, random_state=42)
clf.fit(X_train_emb, y_train)

# 5) prédiction
y_pred = clf.predict(X_test_emb)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

C:\Users\speed\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\speed\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading w

Accuracy: 0.7688338493292054
              precision    recall  f1-score   support

           0       0.78      0.65      0.71       124
           1       0.77      0.91      0.83       559
           2       0.75      0.56      0.64       286

    accuracy                           0.77       969
   macro avg       0.77      0.70      0.73       969
weighted avg       0.77      0.77      0.76       969



# **Part 5 : Classifier 4.0**

## **5.1 - Import Data Set**

Utilisation de df_train_original et df_test_original

## **5.2 - Data Cleaning**

### Text Cleaning

In [4]:
def clean_text_bert(text):
    text = str(text).lower().strip()
    return text

In [ ]:
df_train["sentence_clean"] = df_train_original["sentence"].apply(clean_text_bert)
df_test["sentence_clean"] = df_test_original["sentence"].apply(clean_text_bert)

## **5.3 - Fine-Tuning BERT Transformers Classifier**

In [9]:
# pip install transformers datasets evaluate accelerate torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import numpy as np
import evaluate

# 1) Préparer les données
train_ds = Dataset.from_pandas(df_train[["sentence", "label"]])
test_ds = Dataset.from_pandas(df_test[["sentence", "label"]])

# 2) Tokenizer + modèle
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

# Colonnes utiles
cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

# Padding dynamique
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 3) Métrique
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

# 4) Entraînement
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3966.32it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 969/969 [00:00<00:00, 12397.94 examples/s]
C:\Users\speed\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\

Epoch,Training Loss,Validation Loss,Accuracy
1,0.417663,0.393916,0.842105


TrainOutput(global_step=485, training_loss=0.5545138880149605, metrics={'train_runtime': 619.7747, 'train_samples_per_second': 6.255, 'train_steps_per_second': 0.783, 'total_flos': 52769018827908.0, 'train_loss': 0.5545138880149605, 'epoch': 1.0})

In [12]:
pred_output = trainer.predict(test_ds)
print(pred_output.metrics)

C:\Users\speed\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'test_loss': 0.3939160704612732, 'test_accuracy': 0.8421052631578947, 'test_runtime': 38.4047, 'test_samples_per_second': 25.231, 'test_steps_per_second': 3.177}
